# CKD Deepfake — Colab Setup

Jalankan notebook ini di **awal setiap Colab session**.

**Urutan:**
1. Mount Google Drive
2. Clone / pull repo dari GitHub
3. Install dependencies
4. Download teacher weights ke Drive (sekali saja)
5. Copy hot data dari Drive ke disk lokal
6. Verifikasi GPU + config loader

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/CKD_Thesis'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 2. Clone Repository dari GitHub

In [ ]:
import os

GITHUB_USER = 'tesisbright123-blip'
REPO_NAME   = 'ckd-deepfake'
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR    = f'/content/{REPO_NAME}'

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log -1 --oneline

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
print('Dependencies installed.')

## 4. Download Teacher Weights ke Drive

**Cukup dijalankan SEKALI.** Cell ini skip otomatis kalau file sudah ada di Drive.

| Model | Sumber | Ukuran |
|-------|--------|--------|
| EfficientNet-B4 | DeepfakeBench GitHub Releases | ~68 MB |
| RECCE | DeepfakeBench GitHub Releases | ~183 MB |
| CLIP ViT-L/14 | CLIPping Google Drive | ~1.7 GB |

In [ ]:
import os

TEACHER_DIR = os.path.join(DRIVE_ROOT, 'checkpoints', 'teachers')
os.makedirs(TEACHER_DIR, exist_ok=True)

# ------------------------------------------------------------------
# EfficientNet-B4  (DeepfakeBench, ~68 MB)
# ------------------------------------------------------------------
effnb4_dst = os.path.join(TEACHER_DIR, 'efficientnet_b4.pth')
if os.path.isfile(effnb4_dst):
    print(f'[skip] efficientnet_b4.pth sudah ada ({os.path.getsize(effnb4_dst)/1e6:.1f} MB)')
else:
    print('[download] efficientnet_b4.pth ...')
    !wget -q --show-progress \
        'https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.1/effnb4_best.pth' \
        -O '{effnb4_dst}'
    print(f'  saved -> {effnb4_dst}')

# ------------------------------------------------------------------
# RECCE  (DeepfakeBench, ~183 MB)
# ------------------------------------------------------------------
recce_dst = os.path.join(TEACHER_DIR, 'recce.pth')
if os.path.isfile(recce_dst):
    print(f'[skip] recce.pth sudah ada ({os.path.getsize(recce_dst)/1e6:.1f} MB)')
else:
    print('[download] recce.pth ...')
    !wget -q --show-progress \
        'https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.1/recce_best.pth' \
        -O '{recce_dst}'
    print(f'  saved -> {recce_dst}')

# ------------------------------------------------------------------
# CLIP ViT-L/14 (CLIPping, ~1.7 GB via gdown)
# ------------------------------------------------------------------
clip_dst = os.path.join(TEACHER_DIR, 'clip_clipping.pth')
if os.path.isfile(clip_dst):
    print(f'[skip] clip_clipping.pth sudah ada ({os.path.getsize(clip_dst)/1e6:.1f} MB)')
else:
    print('[download] clip_clipping.pth (besar, ~1.7 GB) ...')
    !pip install -q gdown
    import gdown
    gdown.download(
        id='1jtkBoIuMw5wrooTv-anh668G3_Xt-UnX',
        output=clip_dst,
        quiet=False,
        fuzzy=True,
    )
    print(f'  saved -> {clip_dst}')

# ------------------------------------------------------------------
# Ringkasan
# ------------------------------------------------------------------
print('\n=== Teacher weights ===')
for fname in ['efficientnet_b4.pth', 'recce.pth', 'clip_clipping.pth']:
    fpath = os.path.join(TEACHER_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  OK  {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
    else:
        print(f'  MISSING  {fname}')

## 5. Copy Hot Data ke Disk Lokal

Drive I/O lambat saat training. Copy extracted faces + split CSVs + soft labels
ke `/content/ckd_local/` untuk I/O ~10-20x lebih cepat.

Idempotent — skip kalau sudah ada.

In [ ]:
import os, shutil, time

LOCAL_ROOT = '/content/ckd_local'
os.makedirs(LOCAL_ROOT, exist_ok=True)

SUBDIRS = ['datasets', 'soft_labels']

for sub in SUBDIRS:
    src = os.path.join(DRIVE_ROOT, sub)
    dst = os.path.join(LOCAL_ROOT, sub)
    if not os.path.isdir(src):
        print(f'[skip] {src} belum ada (isi dulu dengan scripts/01-02)')
        continue
    if os.path.isdir(dst):
        print(f'[skip] {dst} sudah ada')
        continue
    t0 = time.time()
    print(f'[copy] {src} -> {dst} ...')
    shutil.copytree(src, dst)
    print(f'  selesai {time.time()-t0:.1f}s')

!du -sh {LOCAL_ROOT}/* 2>/dev/null || echo '(kosong — belum ada data)'

## 6. Verifikasi GPU & Config

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: GPU tidak terdeteksi. Runtime > Change runtime type > T4 GPU')

In [ ]:
import sys
sys.path.insert(0, '/content/ckd-deepfake')

from src.utils.config import load_config

cfg = load_config('configs/default.yaml')
print('Config OK')
print(f'  drive_root : {cfg["paths"]["drive_root"]}')
print(f'  teachers   : {[t["name"] for t in cfg["teacher"]["models"]]}')  
print(f'  student    : {cfg["student"]["architecture"]}')
print()
print('Setup selesai. Siap training.')